In [1]:

from dotenv import load_dotenv, find_dotenv
import os

from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_openai import ChatOpenAI



from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.caches import InMemoryCache
from langchain_core.messages import (
    AIMessage, 
    HumanMessage, 
    SystemMessage,
    ToolMessage
)
from langchain_core.prompts import (
    ChatPromptTemplate,
    PromptTemplate,
    SystemMessagePromptTemplate,
    AIMessagePromptTemplate,
    HumanMessagePromptTemplate,
    MessagesPlaceholder
)

import langchain
from pydantic import BaseModel, Field
from typing import List

from langchain_community.chat_message_histories.in_memory import ChatMessageHistory
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import *
from langchain_text_splitters import CharacterTextSplitter


from langchain.tools import tool
from langchain.agents import create_agent

from langsmith import traceable


USER_AGENT environment variable not set, consider setting it to identify your requests.
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
load_dotenv(find_dotenv())
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [3]:

from langchain_tavily import TavilySearch
from pydantic import BaseModel, Field
from typing import List
from langchain.agents import create_agent
from langchain.tools import tool

In [4]:
MAX_ITERATIONS = 10
MODEL = "qwen3:1.7b"

In [5]:

@tool
def get_product_price(product: str) -> float:
    """Look up the price of a product in the catalog."""
    print(f"    >> Executing get_product_price(product='{product}')")
    prices = {"laptop": 1299.99, "headphones": 149.95, "keyboard": 89.50}
    return prices.get(product, 0)


@tool
def apply_discount(price: float, discount_tier: str) -> float:
    """Apply a discount tier to a price and return the final price.
    Available tiers: bronze, silver, gold."""
    print(f"    >> Executing apply_discount(price={price}, discount_tier='{discount_tier}')")
    discount_percentages = {"bronze": 5, "silver": 12, "gold": 23}
    discount = discount_percentages.get(discount_tier, 0)
    return round(price * (1 - discount / 100), 2)


In [6]:
def run_agent(question: str):
    tools = [get_product_price, apply_discount]
    tools_dict = {t.name: t for t in tools}

    llm=ChatOpenAI(model="gpt-5-nano", openai_api_key=OPENAI_API_KEY)
    llm_with_tools = llm.bind_tools(tools)

    print(f"Question: {question}")
    print("=" * 60)

    messages = [
        SystemMessage(
            content=(
                "You are a helpful shopping assistant. "
                "You have access to a product catalog tool "
                "and a discount tool.\n\n"
                "STRICT RULES — you must follow these exactly:\n"
                "1. NEVER guess or assume any product price. "
                "You MUST call get_product_price first to get the real price.\n"
                "2. Only call apply_discount AFTER you have received "
                "a price from get_product_price. Pass the exact price "
                "returned by get_product_price — do NOT pass a made-up number.\n"
                "3. NEVER calculate discounts yourself using math. "
                "Always use the apply_discount tool.\n"
                "4. If the user does not specify a discount tier, "
                "ask them which tier to use — do NOT assume one."
            )
        ),
        HumanMessage(content=question),
    ]

    for iteration in range(1, MAX_ITERATIONS + 1):
        print(f"\n--- Iteration {iteration} ---")

        ai_message = llm_with_tools.invoke(messages)

        tool_calls = ai_message.tool_calls

        # If no tool calls, this is the final answer
        if not tool_calls:
            print(f"\nFinal Answer: {ai_message.content}")
            return ai_message.content

        # Process only the FIRST tool call — force one tool per iteration
        tool_call = tool_calls[0]
        tool_name = tool_call.get("name")
        tool_args = tool_call.get("args", {})
        tool_call_id = tool_call.get("id")

        print(f"  [Tool Selected] {tool_name} with args: {tool_args}")

        tool_to_use = tools_dict.get(tool_name)
        if tool_to_use is None:
            raise ValueError(f"Tool '{tool_name}' not found")

        observation = tool_to_use.invoke(tool_args)

        print(f"  [Tool Result] {observation}")

        messages.append(ai_message)
        messages.append(
            ToolMessage(content=str(observation), tool_call_id=tool_call_id)
        )

    print("ERROR: Max iterations reached without a final answer")
    return None

In [7]:
result = run_agent("What is the price of a laptop after applying a gold discount?")

Question: What is the price of a laptop after applying a gold discount?

--- Iteration 1 ---
  [Tool Selected] get_product_price with args: {'product': 'laptop'}
    >> Executing get_product_price(product='laptop')
  [Tool Result] 1299.99

--- Iteration 2 ---
  [Tool Selected] apply_discount with args: {'price': 1299.99, 'discount_tier': 'gold'}
    >> Executing apply_discount(price=1299.99, discount_tier='gold')
  [Tool Result] 1000.99

--- Iteration 3 ---

Final Answer: The price of the laptop after applying a gold discount is $1000.99.
